# GLM Distribution Diagnostics

This notebook tests whether a generalized linear model-style likelihood improves the strong linear/additive baselines. The goal is not to force every GLM family onto the problem, but to compare valid continuous-response choices and record which distribution assumptions are compatible with `TVT`, endpoint residuals, and target-zone deltas.

## 1. Setup

In [1]:
"Configure imports, paths, and deterministic settings."

from pathlib import Path
import json
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import GammaRegressor, HuberRegressor, PoissonRegressor, QuantileRegressor, Ridge, TweedieRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.validation import group_well_folds
from src.io.loaders import discover_wells, load_horizontal_well
from src.utils.run_bundle import RunBundle

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

RUN_TAG = "glm_distribution_diagnostics"
RUN_DATE = "2026-06-22"
SEED = 42
N_SPLITS = 3
MAX_TARGET_ROWS_PER_WELL = 140
MAX_TRAIN_ROWS_PER_FOLD = 60000
RESIDUAL_SAMPLE_PER_FOLD = 1200

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ARTIFACT_DIR / "tables"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"

for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")

## 2. Feature Construction

In [2]:
"Define notebook-local slope-core feature construction helpers."


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def deterministic_row_cap(frame, max_rows):
    if len(frame) <= max_rows:
        return frame
    positions = np.linspace(0, len(frame) - 1, max_rows).astype(int)
    return frame.iloc[positions].copy()


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def build_slope_core_features_for_well(well, directory, max_rows):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    target_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    known_mask = horizontal["TVT_input"].notna() & horizontal["TVT"].notna()
    if int(target_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{window}": window_slope(md, tvt_input, last_known_pos, window) for window in slope_windows}
    z_slopes = {f"z_slope_last_{window}": window_slope(md, z, last_known_pos, window) for window in slope_windows}

    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()

    target_positions = deterministic_sample_positions(target_mask, max_rows)
    all_target_positions = np.flatnonzero(target_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)

    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "MD": float(md.iloc[pos]),
            "Z": float(z.iloc[pos]),
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "last_known_tvt": last_known_tvt,
            "last_known_md": last_known_md,
            "last_known_z": last_known_z,
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "TVT": float(horizontal["TVT"].iloc[pos]),
            "endpoint_hold_pred": last_known_tvt,
        }
        row["delta_tvt_from_boundary"] = row["TVT"] - last_known_tvt
        row["endpoint_residual"] = row["TVT"] - row["endpoint_hold_pred"]
        for name, value in tvt_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
            row[f"{name}_projected_delta"] = row[name] * distance_md if np.isfinite(row[name]) else np.nan
        for name, value in z_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


slope_features = ["tvt_slope_last_10", "tvt_slope_last_25", "tvt_slope_last_50", "tvt_slope_last_100", "z_slope_last_25", "z_slope_last_50", "z_slope_last_100"]
slope_projection_features = ["tvt_slope_last_10_projected_delta", "tvt_slope_last_25_projected_delta", "tvt_slope_last_50_projected_delta", "tvt_slope_last_100_projected_delta"]
feature_columns = [
    "distance_from_boundary_md",
    "eval_progress",
    "md_norm",
    "dZ_from_boundary",
    "horizontal_dist_from_boundary",
] + slope_features + slope_projection_features + ["last_known_tvt"]

print({"n_features": len(feature_columns)})

{'n_features': 17}


In [3]:
"Build the sampled target-zone feature frame and distribution diagnostics."

start = time.time()
train_wells = discover_wells(TRAIN_DIR)
frames = [build_slope_core_features_for_well(well, TRAIN_DIR, MAX_TARGET_ROWS_PER_WELL) for well in train_wells]
feature_frame = pd.concat([frame for frame in frames if not frame.empty], ignore_index=True)
build_seconds = time.time() - start

target_diagnostics = pd.DataFrame([
    {
        "target": "TVT",
        "min": float(feature_frame["TVT"].min()),
        "max": float(feature_frame["TVT"].max()),
        "mean": float(feature_frame["TVT"].mean()),
        "std": float(feature_frame["TVT"].std()),
        "skew": float(skew(feature_frame["TVT"].to_numpy(float), nan_policy="omit")),
        "excess_kurtosis": float(kurtosis(feature_frame["TVT"].to_numpy(float), nan_policy="omit")),
        "positive_fraction": float((feature_frame["TVT"] > 0).mean()),
        "bounded_0_1_fraction": float(((feature_frame["TVT"] > 0) & (feature_frame["TVT"] < 1)).mean()),
    },
    {
        "target": "delta_tvt_from_boundary",
        "min": float(feature_frame["delta_tvt_from_boundary"].min()),
        "max": float(feature_frame["delta_tvt_from_boundary"].max()),
        "mean": float(feature_frame["delta_tvt_from_boundary"].mean()),
        "std": float(feature_frame["delta_tvt_from_boundary"].std()),
        "skew": float(skew(feature_frame["delta_tvt_from_boundary"].to_numpy(float), nan_policy="omit")),
        "excess_kurtosis": float(kurtosis(feature_frame["delta_tvt_from_boundary"].to_numpy(float), nan_policy="omit")),
        "positive_fraction": float((feature_frame["delta_tvt_from_boundary"] > 0).mean()),
        "bounded_0_1_fraction": float(((feature_frame["delta_tvt_from_boundary"] > 0) & (feature_frame["delta_tvt_from_boundary"] < 1)).mean()),
    },
    {
        "target": "endpoint_residual",
        "min": float(feature_frame["endpoint_residual"].min()),
        "max": float(feature_frame["endpoint_residual"].max()),
        "mean": float(feature_frame["endpoint_residual"].mean()),
        "std": float(feature_frame["endpoint_residual"].std()),
        "skew": float(skew(feature_frame["endpoint_residual"].to_numpy(float), nan_policy="omit")),
        "excess_kurtosis": float(kurtosis(feature_frame["endpoint_residual"].to_numpy(float), nan_policy="omit")),
        "positive_fraction": float((feature_frame["endpoint_residual"] > 0).mean()),
        "bounded_0_1_fraction": float(((feature_frame["endpoint_residual"] > 0) & (feature_frame["endpoint_residual"] < 1)).mean()),
    },
])

print({
    "wells": len(train_wells),
    "sample_rows": len(feature_frame),
    "build_seconds": round(build_seconds, 2),
})
target_diagnostics

21:31:43 | INFO     | src.io.loaders | Discovered 773 wells in C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw\train


{'wells': 773, 'sample_rows': 108220, 'build_seconds': 51.73}


,target,min,max,mean,std,skew,excess_kurtosis,positive_fraction,bounded_0_1_fraction
0,TVT,10039.19,12893.88,11552.472778,619.737518,0.361622,-0.828974,1.000000,0.000000
1,delta_tvt_from_boundary,-103.78,98.83,1.416659,15.523782,0.362655,4.087509,0.532831,0.048235
2,endpoint_residual,-103.78,98.83,1.416659,15.523782,0.362655,4.087509,0.532831,0.048235


## 3. Distribution Families

Classic GLMs are only useful when the response geometry matches the family. `TVT` is a positive continuous scalar, so Gamma/Tweedie-style positive continuous models are admissible on raw `TVT`. Endpoint residuals and boundary deltas are continuous and may be signed, so Gaussian, robust Gaussian, and quantile-style losses are more natural. Beta regression is for values in `(0, 1)`, and Dirichlet regression is for vectors of proportions that sum to one, so neither is fit as a candidate here.

In [4]:
"Define GLM and robust-linear candidates with explicit target geometry."


def make_preprocessor():
    return ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            feature_columns,
        )
    ], remainder="drop", sparse_threshold=0.0)


def make_pipeline(estimator):
    return Pipeline([
        ("features", make_preprocessor()),
        ("model", estimator),
    ])


candidate_specs = [
    {"candidate": "gaussian_delta_ridge_a30", "family": "gaussian", "target_mode": "delta_boundary", "estimator": Ridge(alpha=30.0), "notes": "Continuous signed delta with squared-error Gaussian mean model."},
    {"candidate": "gaussian_delta_tweedie_p0_a1", "family": "tweedie_power0_gaussian", "target_mode": "delta_boundary", "estimator": TweedieRegressor(power=0, alpha=1.0, link="identity", max_iter=1000), "notes": "Tweedie power 0 is the normal-distribution GLM case for signed continuous deltas."},
    {"candidate": "robust_delta_huber", "family": "robust_gaussian", "target_mode": "delta_boundary", "estimator": HuberRegressor(alpha=0.0001, epsilon=1.35, max_iter=1000), "notes": "Robust continuous signed delta model for heavy residual tails."},
    {"candidate": "median_delta_quantile_a0p001", "family": "quantile", "target_mode": "delta_boundary", "estimator": QuantileRegressor(quantile=0.5, alpha=0.001, solver="highs"), "notes": "Median conditional delta model; robust but can underfit the mean RMSE objective."},
    {"candidate": "raw_tvt_poisson_log_a1", "family": "poisson_quasi", "target_mode": "raw_tvt", "estimator": PoissonRegressor(alpha=1.0, max_iter=1000), "notes": "Positive-response log-link model; TVT is not count data, so this is a quasi-likelihood diagnostic only."},
    {"candidate": "raw_tvt_gamma_log_a1", "family": "gamma", "target_mode": "raw_tvt", "estimator": GammaRegressor(alpha=1.0, max_iter=1000), "notes": "Positive continuous raw TVT with Gamma log-link."},
    {"candidate": "raw_tvt_tweedie_p1p5_a1", "family": "tweedie_positive", "target_mode": "raw_tvt", "estimator": TweedieRegressor(power=1.5, alpha=1.0, link="log", max_iter=1000), "notes": "Positive continuous compound-Poisson/Gamma-style TVT diagnostic."},
    {"candidate": "raw_tvt_tweedie_p2_a1", "family": "gamma_tweedie", "target_mode": "raw_tvt", "estimator": TweedieRegressor(power=2.0, alpha=1.0, link="log", max_iter=1000), "notes": "Tweedie power 2 matches Gamma variance structure for positive raw TVT."},
    {"candidate": "shifted_delta_gamma_log_a1", "family": "shifted_gamma", "target_mode": "shifted_delta", "estimator": GammaRegressor(alpha=1.0, max_iter=1000), "notes": "Positive shifted delta; mathematically admissible but shift-dependent and less interpretable."},
    {"candidate": "shifted_delta_tweedie_p1p5_a1", "family": "shifted_tweedie_positive", "target_mode": "shifted_delta", "estimator": TweedieRegressor(power=1.5, alpha=1.0, link="log", max_iter=1000), "notes": "Positive shifted delta with Tweedie log-link; diagnostic only because the shift is fold-specific."},
]

family_validity = pd.DataFrame([
    {"family": "Gaussian", "fit_status": "fit", "reason": "Valid for signed continuous delta/residual targets."},
    {"family": "Huber", "fit_status": "fit", "reason": "Robust continuous-response loss for heavy-tailed residuals."},
    {"family": "Quantile", "fit_status": "fit", "reason": "Valid for conditional median of signed continuous deltas."},
    {"family": "Gamma", "fit_status": "fit_on_positive_targets", "reason": "Valid for positive continuous raw TVT or shifted delta, but not signed residuals."},
    {"family": "Poisson", "fit_status": "diagnostic_only", "reason": "Requires nonnegative mean and is intended for counts; raw TVT is positive but continuous."},
    {"family": "Tweedie", "fit_status": "fit", "reason": "Power 0 covers Gaussian-like deltas; powers 1-2 cover positive skewed responses."},
    {"family": "Beta", "fit_status": "not_fit", "reason": "Requires scalar values strictly in (0, 1); TVT and deltas are depth-unit responses."},
    {"family": "Dirichlet", "fit_status": "not_fit", "reason": "Requires composition vectors summing to one; this competition predicts one scalar TVT per row."},
])

candidate_table = pd.DataFrame([{key: spec[key] for key in ["candidate", "family", "target_mode", "notes"]} for spec in candidate_specs])
print({"candidate_count": len(candidate_specs)})
candidate_table

{'candidate_count': 10}


,candidate,family,target_mode,notes
0,gaussian_delta_ridge_a30,gaussian,delta_boundary,Continuous signed delta with squared-error Gau...
1,gaussian_delta_tweedie_p0_a1,tweedie_power0_gaussian,delta_boundary,Tweedie power 0 is the normal-distribution GLM...
2,robust_delta_huber,robust_gaussian,delta_boundary,Robust continuous signed delta model for heavy...
3,median_delta_quantile_a0p001,quantile,delta_boundary,Median conditional delta model; robust but can...
4,raw_tvt_poisson_log_a1,poisson_quasi,raw_tvt,Positive-response log-link model; TVT is not c...
5,raw_tvt_gamma_log_a1,gamma,raw_tvt,Positive continuous raw TVT with Gamma log-link.
6,raw_tvt_tweedie_p1p5_a1,tweedie_positive,raw_tvt,Positive continuous compound-Poisson/Gamma-sty...
7,raw_tvt_tweedie_p2_a1,gamma_tweedie,raw_tvt,Tweedie power 2 matches Gamma variance structu...
8,shifted_delta_gamma_log_a1,shifted_gamma,shifted_delta,Positive shifted delta; mathematically admissi...
9,shifted_delta_tweedie_p1p5_a1,shifted_tweedie_positive,shifted_delta,Positive shifted delta with Tweedie log-link; ...


## 4. Grouped CV Sweep

In [5]:
"Run grouped well-fold CV across GLM and robust-linear candidates."


def target_values(frame, target_mode, shift_value=None):
    if target_mode == "delta_boundary":
        return frame["delta_tvt_from_boundary"].to_numpy(float)
    if target_mode == "raw_tvt":
        return frame["TVT"].to_numpy(float)
    if target_mode == "shifted_delta":
        return frame["delta_tvt_from_boundary"].to_numpy(float) - shift_value + 1.0
    raise ValueError(f"Unknown target mode: {target_mode}")


def reconstruct_tvt(frame, prediction, target_mode, shift_value=None):
    if target_mode == "delta_boundary":
        return frame["last_known_tvt"].to_numpy(float) + prediction
    if target_mode == "raw_tvt":
        return prediction
    if target_mode == "shifted_delta":
        return frame["last_known_tvt"].to_numpy(float) + prediction + shift_value - 1.0
    raise ValueError(f"Unknown target mode: {target_mode}")


def metric_row(y_true, pred_tvt):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, pred_tvt))),
        "mae": float(mean_absolute_error(y_true, pred_tvt)),
        "r2": float(r2_score(y_true, pred_tvt)),
    }


fit_start = time.time()
fold_rows = []
residual_rows = []
wells = sorted(feature_frame["wellname"].unique())
folds = group_well_folds(wells, n_splits=N_SPLITS)

for fold_idx, (train_fold_wells, valid_fold_wells) in enumerate(folds, start=1):
    train_frame = feature_frame.loc[feature_frame["wellname"].isin(train_fold_wells)].copy()
    valid_frame = feature_frame.loc[feature_frame["wellname"].isin(valid_fold_wells)].copy()
    train_frame = deterministic_row_cap(train_frame, MAX_TRAIN_ROWS_PER_FOLD)
    y_true = valid_frame["TVT"].to_numpy(float)

    endpoint_pred = valid_frame["endpoint_hold_pred"].to_numpy(float)
    endpoint_metrics = metric_row(y_true, endpoint_pred)
    fold_rows.append({"fold": fold_idx, "candidate": "endpoint_hold", "family": "baseline", "target_mode": "baseline", "n_train_rows": 0, "n_valid_rows": len(valid_frame), "shift_value": np.nan, **endpoint_metrics})

    for spec in candidate_specs:
        shift_value = np.nan
        if spec["target_mode"] == "shifted_delta":
            shift_value = float(train_frame["delta_tvt_from_boundary"].min())
        model = make_pipeline(spec["estimator"])
        y_train = target_values(train_frame, spec["target_mode"], shift_value)
        model.fit(train_frame[feature_columns], y_train)
        pred_target = model.predict(valid_frame[feature_columns])
        pred_tvt = reconstruct_tvt(valid_frame, pred_target, spec["target_mode"], shift_value)
        metrics = metric_row(y_true, pred_tvt)
        fold_rows.append({"fold": fold_idx, "candidate": spec["candidate"], "family": spec["family"], "target_mode": spec["target_mode"], "n_train_rows": len(train_frame), "n_valid_rows": len(valid_frame), "shift_value": shift_value, **metrics})
        if spec["candidate"] in {"gaussian_delta_ridge_a30", "robust_delta_huber", "raw_tvt_gamma_log_a1"}:
            sample_idx = np.linspace(0, len(valid_frame) - 1, min(len(valid_frame), RESIDUAL_SAMPLE_PER_FOLD)).astype(int)
            residual_rows.append(pd.DataFrame({
                "fold": fold_idx,
                "candidate": spec["candidate"],
                "family": spec["family"],
                "target_mode": spec["target_mode"],
                "wellname": valid_frame.iloc[sample_idx]["wellname"].to_numpy(),
                "row_idx": valid_frame.iloc[sample_idx]["row_idx"].to_numpy(),
                "distance_from_boundary_md": valid_frame.iloc[sample_idx]["distance_from_boundary_md"].to_numpy(float),
                "y_true": y_true[sample_idx],
                "y_pred": pred_tvt[sample_idx],
                "residual": y_true[sample_idx] - pred_tvt[sample_idx],
            }))

fit_seconds = time.time() - fit_start
fold_metrics = pd.DataFrame(fold_rows)
summary = (
    fold_metrics.groupby(["candidate", "family", "target_mode"], as_index=False)
    .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean"), n_valid_rows=("n_valid_rows", "sum"), n_train_rows_mean=("n_train_rows", "mean"))
    .sort_values("rmse_mean")
)
residual_sample = pd.concat(residual_rows, ignore_index=True)

print({"fit_seconds": round(fit_seconds, 2), "fold_rows": len(fold_metrics)})
summary

{'fit_seconds': 877.91, 'fold_rows': 33}


,candidate,family,target_mode,rmse_mean,rmse_std,mae_mean,r2_mean,n_valid_rows,n_train_rows_mean
8,robust_delta_huber,robust_gaussian,delta_boundary,14.792843,0.471640,10.392285,0.999429,108220,60000.0
1,gaussian_delta_ridge_a30,gaussian,delta_boundary,14.839828,0.455638,10.410899,0.999426,108220,60000.0
3,median_delta_quantile_a0p001,quantile,delta_boundary,14.884181,0.519474,10.404815,0.999422,108220,60000.0
10,shifted_delta_tweedie_p1p5_a1,shifted_tweedie_positive,shifted_delta,15.000768,0.466424,10.562165,0.999413,108220,60000.0
9,shifted_delta_gamma_log_a1,shifted_gamma,shifted_delta,15.035638,0.471780,10.599294,0.999410,108220,60000.0
2,gaussian_delta_tweedie_p0_a1,tweedie_power0_gaussian,delta_boundary,15.037580,0.466856,10.600533,0.999410,108220,60000.0
0,endpoint_hold,baseline,baseline,15.578175,0.651555,10.905266,0.999367,108220,0.0
5,raw_tvt_poisson_log_a1,poisson_quasi,raw_tvt,22.178149,2.268262,16.476272,0.998715,108220,60000.0
6,raw_tvt_tweedie_p1p5_a1,tweedie_positive,raw_tvt,22.558418,3.004583,16.903951,0.998668,108220,60000.0
4,raw_tvt_gamma_log_a1,gamma,raw_tvt,311.411565,9.099637,270.715211,0.747128,108220,60000.0


## 5. Artifacts And Diagnostics

In [6]:
"Export tables and diagnostic figures."

fold_metrics_path = TABLE_DIR / f"model__glm_distribution_fold_metrics__{RUN_DATE}.csv"
summary_path = TABLE_DIR / f"model__glm_distribution_summary__{RUN_DATE}.csv"
candidate_path = TABLE_DIR / f"model__glm_distribution_candidates__{RUN_DATE}.csv"
validity_path = TABLE_DIR / f"model__glm_distribution_family_validity__{RUN_DATE}.csv"
target_diag_path = TABLE_DIR / f"model__glm_distribution_target_diagnostics__{RUN_DATE}.csv"
residual_path = TABLE_DIR / f"model__glm_distribution_residual_sample__{RUN_DATE}.csv"

fold_metrics.to_csv(fold_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
candidate_table.to_csv(candidate_path, index=False)
family_validity.to_csv(validity_path, index=False)
target_diagnostics.to_csv(target_diag_path, index=False)
residual_sample.to_csv(residual_path, index=False)

fig_paths = {}
plot_summary = summary.sort_values("rmse_mean", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#4C78A8" if family != "baseline" else "#F58518" for family in plot_summary["family"]]
ax.barh(plot_summary["candidate"], plot_summary["rmse_mean"], color=colors)
ax.set_title("GLM And Robust Linear Candidate RMSE")
ax.set_xlabel("grouped target-zone RMSE")
ax.set_ylabel("")
fig.tight_layout()
fig_paths["rmse_comparison"] = FIGURE_DIR / f"model__glm_distribution_rmse_comparison__{RUN_DATE}.png"
fig.savefig(fig_paths["rmse_comparison"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
for candidate, group in fold_metrics.groupby("candidate"):
    if candidate in set(summary.head(4)["candidate"]) or candidate == "endpoint_hold":
        ax.plot(group["fold"], group["rmse"], marker="o", label=candidate)
ax.set_title("GLM Candidate Fold RMSE")
ax.set_xlabel("fold")
ax.set_ylabel("RMSE")
ax.legend(fontsize=7)
fig.tight_layout()
fig_paths["fold_rmse"] = FIGURE_DIR / f"model__glm_distribution_fold_rmse__{RUN_DATE}.png"
fig.savefig(fig_paths["fold_rmse"], dpi=160)
plt.close(fig)

best_candidate = str(summary.iloc[0]["candidate"])
if best_candidate in residual_sample["candidate"].unique():
    best_residuals = residual_sample.loc[residual_sample["candidate"] == best_candidate]
else:
    reference_candidate = "gaussian_delta_ridge_a30"
    best_residuals = residual_sample.loc[residual_sample["candidate"] == reference_candidate]
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_residuals["y_pred"], best_residuals["residual"], s=6, alpha=0.35, color="#54A24B")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title(f"Residuals for {best_candidate}")
ax.set_xlabel("predicted TVT")
ax.set_ylabel("true - predicted")
fig.tight_layout()
fig_paths["residuals"] = FIGURE_DIR / f"model__glm_distribution_residuals__{RUN_DATE}.png"
fig.savefig(fig_paths["residuals"], dpi=160)
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, target in zip(axes, ["TVT", "delta_tvt_from_boundary", "endpoint_residual"]):
    ax.hist(feature_frame[target].to_numpy(float), bins=45, color="#4C78A8", alpha=0.85)
    ax.set_title(target)
    ax.set_ylabel("rows")
fig.tight_layout()
fig_paths["target_distributions"] = FIGURE_DIR / f"model__glm_distribution_target_histograms__{RUN_DATE}.png"
fig.savefig(fig_paths["target_distributions"], dpi=160)
plt.close(fig)

{key: str(value) for key, value in fig_paths.items()}

{'rmse_comparison': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__glm_distribution_rmse_comparison__2026-06-22.png',
 'fold_rmse': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__glm_distribution_fold_rmse__2026-06-22.png',
 'residuals': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__glm_distribution_residuals__2026-06-22.png',
 'target_distributions': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__glm_distribution_target_histograms__2026-06-22.png'}

In [7]:
"Write report, update model comparison, and create run bundle."

best = summary.iloc[0]
endpoint = summary.loc[summary["candidate"] == "endpoint_hold"].iloc[0]
gaussian = summary.loc[summary["candidate"] == "gaussian_delta_ridge_a30"].iloc[0]
gam_reference_rmse = 14.868732354995856
kaggle_gam_public = 15.055

findings = pd.DataFrame([
    {"topic": "best_glm_candidate", "finding": f"Best GLM/robust candidate is {best.candidate} with sampled grouped RMSE {best.rmse_mean:.4f}.", "modeling_implication": "Use this result to decide whether GLM likelihood changes beat the current GAM spline baseline."},
    {"topic": "gaussian_delta_reference", "finding": f"Gaussian Ridge on slope-core delta scored RMSE {gaussian.rmse_mean:.4f}, compared with endpoint hold {endpoint.rmse_mean:.4f}.", "modeling_implication": "A simple continuous signed-delta model remains a strong reference for later Bayesian likelihood choices."},
    {"topic": "classic_glm_validity", "finding": "Beta and Dirichlet families are not fit because TVT is a scalar depth response, not a bounded probability or composition vector.", "modeling_implication": "Prefer Gaussian/Student-t/robust or positive continuous families over count/proportion families for Bayesian modeling."},
    {"topic": "gam_reference", "finding": f"Current submitted GAM spline has local diagnostic RMSE {gam_reference_rmse:.4f} and public LB {kaggle_gam_public:.3f}.", "modeling_implication": "A GLM candidate should clear this local hurdle before becoming a Kaggle submission candidate."},
])
findings_path = TABLE_DIR / f"model__glm_distribution_findings__{RUN_DATE}.csv"
findings.to_csv(findings_path, index=False)

report_path = REPORT_DIR / f"model__glm_distribution_diagnostics__{RUN_DATE}.md"
report_lines = ["# GLM Distribution Diagnostics", "", f"Date: {RUN_DATE}", "", "## Key Findings"]
for row in findings.itertuples(index=False):
    report_lines.extend([f"- **{row.topic}**: {row.finding}", f"  - Modeling implication: {row.modeling_implication}"])
report_lines.extend([
    "",
    "## Best Candidate",
    f"- Candidate: `{best.candidate}`",
    f"- Family: `{best.family}`",
    f"- Target mode: `{best.target_mode}`",
    f"- RMSE mean/std: {best.rmse_mean:.4f} / {best.rmse_std:.4f}",
    "",
    "## Artifacts",
    f"- Fold metrics: {fold_metrics_path}",
    f"- Summary: {summary_path}",
    f"- Candidate table: {candidate_path}",
    f"- Family validity: {validity_path}",
    f"- Target diagnostics: {target_diag_path}",
    f"- Residual sample: {residual_path}",
    f"- Findings: {findings_path}",
])
for name, fig_path in fig_paths.items():
    report_lines.append(f"- {name}: {fig_path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path)
run_comparison_row = pd.DataFrame([{
    "run_id": "pending_run_bundle",
    "model_family": "glm_distribution",
    "model_name": str(best.candidate),
    "cv_rmse_mean": float(best.rmse_mean),
    "cv_rmse_std": float(best.rmse_std),
    "train_time_s": float(build_seconds + fit_seconds),
    "n_features": int(len(feature_columns)),
    "notes": "GLM and robust-likelihood diagnostics on slope-core target-zone features; notebook-local features",
}])
comparison = comparison.loc[comparison["run_id"] != "pending_run_bundle"]
comparison = pd.concat([comparison, run_comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

config = {
    "task": "glm_distribution_diagnostics",
    "seed": SEED,
    "n_splits": N_SPLITS,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD,
    "candidate_count": int(len(candidate_specs)),
    "best_candidate": str(best.candidate),
    "best_family": str(best.family),
    "best_target_mode": str(best.target_mode),
    "note": "GLM/robust-likelihood diagnostics before Bayesian distribution choices.",
}
with RunBundle(RUN_TAG, config, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_metric("sample_rows", float(len(feature_frame)))
    rb.add_metric("candidate_count", float(len(candidate_specs)))
    rb.add_metric("best_rmse", float(best.rmse_mean))
    rb.add_metric("best_rmse_std", float(best.rmse_std))
    rb.add_metric("endpoint_hold_rmse", float(endpoint.rmse_mean))
    rb.add_metric("best_minus_endpoint", float(best.rmse_mean - endpoint.rmse_mean))
    rb.add_metric("best_minus_gam_reference", float(best.rmse_mean - gam_reference_rmse))
    for output_path in [fold_metrics_path, summary_path, candidate_path, validity_path, target_diag_path, residual_path, findings_path, report_path, comparison_path]:
        rb.add_output(str(output_path))
    for fig_path in fig_paths.values():
        rb.add_output(str(fig_path))
    rb.set_notes("GLM distribution diagnostic completed. Beta and Dirichlet are logged as invalid target geometries for scalar TVT.")
run_id = rb.run_id
comparison.loc[comparison["run_id"] == "pending_run_bundle", "run_id"] = run_id
comparison.to_csv(comparison_path, index=False)

print({"run_id": run_id, "best_candidate": best.candidate, "best_rmse": round(float(best.rmse_mean), 4), "endpoint_hold_rmse": round(float(endpoint.rmse_mean), 4), "best_minus_gam_reference": round(float(best.rmse_mean - gam_reference_rmse), 4)})
findings

{'run_id': '2026-06-21__2147__glm_distribution_diagnostics__107a7d7', 'best_candidate': 'robust_delta_huber', 'best_rmse': 14.7928, 'endpoint_hold_rmse': 15.5782, 'best_minus_gam_reference': -0.0759}


,topic,finding,modeling_implication
0,best_glm_candidate,Best GLM/robust candidate is robust_delta_hube...,Use this result to decide whether GLM likeliho...
1,gaussian_delta_reference,Gaussian Ridge on slope-core delta scored RMSE...,A simple continuous signed-delta model remains...
2,classic_glm_validity,Beta and Dirichlet families are not fit becaus...,Prefer Gaussian/Student-t/robust or positive c...
3,gam_reference,Current submitted GAM spline has local diagnos...,A GLM candidate should clear this local hurdle...


## Results

Review the executed outputs above. A non-Gaussian GLM family is useful only if it improves the target-zone grouped RMSE while preserving a valid target distribution assumption.